# パターン認識特論 第3回 演習：誤差逆伝播法とニューラルネットワークの訓練

講義スライド（Version 7）の式を NumPy で動かし，「自分の逆伝播が正しい」ことを**数値微分で確かめる**経験をする．
各節の **やってみよう** の `# TODO` を書き換えて実行する．解答は直後のセル．

| 節 | テーマ | スライド |
|---|---|---|
| §1 | 重み1つの数値例と学習率 | p.4 |
| §2 | 2層の鎖の逆伝播（手順どおり） | p.5–9 |
| §3 | NumPy で全結合 MLP の逆伝播と勾配チェック | p.5–9 |
| §4 | 小さな自動微分（計算グラフ） | p.10–12 |
| §5 | SGD・Momentum・Adam の軌跡 | p.13–15 |
| §6 | 学習率・スケジュール・クリッピング | p.16–17 |
| §7 | 勾配消失の実験 | p.17 |
| §8 | （任意）PyTorch の autograd | 付録B |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def sigmoid(u):
    return 1.0 / (1.0 + np.exp(-u))

def relu(u):
    return np.maximum(0.0, u)

## §1 重み1つの数値例と学習率（p.4）

$\hat y = wx$，$L=\frac{1}{2}(\hat y-y)^2$，$x=2$，$y=4$，$w=1$．勾配は $\partial L/\partial w=(\hat y-y)x$．

In [ ]:
x, y = 2.0, 4.0

def run(eta, w=1.0, steps=8):
    hist = []
    for t in range(steps):
        yhat = w * x
        L = 0.5 * (yhat - y) ** 2
        g = (yhat - y) * x
        hist.append((t, w, L, g))
        w = w - eta * g
    return hist

for t, w, L, g in run(0.1):
    print(f"step {t}: w={w:6.3f}  L={L:7.3f}  dL/dw={g:7.3f}")

**やってみよう 1-1（p.4 のクイズ）**　学習率 $\eta$ を 0.1, 0.2, 0.25, 0.3, 1.0 に変えて損失の推移を描く．どこから発散するか．
（この問題では $w \leftarrow w - \eta x (wx-y) = (1-\eta x^2) w + \eta x y$ なので，$|1-\eta x^2|<1$，つまり $\eta<0.5$ が収束の条件．$\eta=0.25$ で 1 回で最適解に着く．）

In [ ]:
plt.figure(figsize=(6, 3.6))
for eta in (0.1, 0.2, 0.25, 0.3, 1.0):      # TODO: 値を変えて試す
    Ls = [h[2] for h in run(eta, steps=8)]
    plt.semilogy(Ls, marker='o', label=f'eta={eta}')
plt.xlabel('step'); plt.ylabel('L (log)'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## §2 2層の鎖の逆伝播（p.5–9）

1 次元の 2 層ネットワーク：$x \to u_1=w_1x \to z_1=\mathrm{ReLU}(u_1) \to u_2=w_2z_1 \to p=\sigma(u_2) \to L=\mathrm{BCE}(p,y)$．
$x=1$，$w_1=0.5$，$w_2=-1$，$y=1$．p.9 の手順（①順伝播 ②$\delta^{(2)}=p-y$ ③再帰 ④重みの勾配）をそのまま書く．

In [ ]:
def chain_forward_backward(w1, w2, x=1.0, y=1.0):
    # ① 順伝播（中間値を保持）
    u1 = w1 * x
    z1 = relu(u1)
    u2 = w2 * z1
    p = sigmoid(u2)
    L = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    # ② 出力層の誤差信号
    d2 = p - y
    # ③ 再帰：δ^(1) = h'(u1) · w2 · δ^(2)
    h1p = 1.0 if u1 > 0 else 0.0
    d1 = h1p * w2 * d2
    # ④ 重みの勾配：δ × 入力側の活性
    gw2 = d2 * z1
    gw1 = d1 * x
    return dict(u1=u1, z1=z1, u2=u2, p=p, L=L, d2=d2, d1=d1, gw1=gw1, gw2=gw2)

r = chain_forward_backward(0.5, -1.0)
for k, v in r.items():
    print(f"{k:4s} = {v: .4f}")

**数値微分と照合**　$w_1$，$w_2$ を $\pm\varepsilon$ 動かして $L$ の変化から勾配を測り，上の `gw1`，`gw2` と一致するか確かめる．

In [ ]:
def num_grad_chain(w1, w2, eps=1e-6):
    L = lambda a, b: chain_forward_backward(a, b)['L']
    return ((L(w1 + eps, w2) - L(w1 - eps, w2)) / (2 * eps),
            (L(w1, w2 + eps) - L(w1, w2 - eps)) / (2 * eps))

print("逆伝播  :", r['gw1'], r['gw2'])
print("数値微分:", num_grad_chain(0.5, -1.0))

**やってみよう 2-1**　$w_1=-0.5$ にすると何が起きるか（p.8 のクイズ：ReLU の負側）．誤差信号と勾配を見る．

In [ ]:
r2 = chain_forward_backward(-0.5, -1.0)   # TODO: 実行して d1, gw1, gw2 を確認
print({k: round(float(v), 4) for k, v in r2.items()})

In [ ]:
# 解答：u1 = −0.5 < 0 なので z1 = 0，h'(u1) = 0 → δ^(1) = 0．両方の勾配が 0 になり，w1 は何回更新しても動かない（dying ReLU）．
print("d1 =", r2['d1'], " gw1 =", r2['gw1'], " gw2 =", r2['gw2'])

## §3 NumPy で全結合 MLP の逆伝播と勾配チェック（p.5–9）

ベクトル版の式をそのまま実装する（ミニバッチは行が標本．したがって式の転置の向きに注意）．

- 順伝播：$U^{(l)} = Z^{(l-1)} W^{(l)\top} + \boldsymbol{b}^{(l)}$，$Z^{(l)} = h(U^{(l)})$
- 出力層：$\Delta^{(L)} = (P - Y)/N$
- 再帰：$\Delta^{(l)} = h'(U^{(l)}) \odot \left(\Delta^{(l+1)} W^{(l+1)}\right)$
- 勾配：$\partial L/\partial W^{(l)} = \Delta^{(l)\top} Z^{(l-1)}$，$\partial L/\partial \boldsymbol{b}^{(l)} = \sum_n \Delta^{(l)}_n$

In [ ]:
def softmax_rows(U):
    U = U - U.max(axis=1, keepdims=True)
    E = np.exp(U)
    return E / E.sum(axis=1, keepdims=True)

def init_mlp(sizes, scale=1.0):
    P = {}
    for l in range(1, len(sizes)):
        P[f'W{l}'] = rng.normal(size=(sizes[l], sizes[l - 1])) * scale / np.sqrt(sizes[l - 1])
        P[f'b{l}'] = np.zeros(sizes[l])
    return P

def mlp_forward(P, X):
    L = len(P) // 2
    cache = [(None, X)]
    Z = X
    for l in range(1, L + 1):
        U = Z @ P[f'W{l}'].T + P[f'b{l}']
        Z = relu(U) if l < L else U       # 最後の層はロジット
        cache.append((U, Z))
    return Z, cache

def mlp_loss(P, X, y):
    logits, _ = mlp_forward(P, X)
    Pr = softmax_rows(logits)
    return -np.mean(np.log(Pr[np.arange(len(y)), y] + 1e-12))

def mlp_backward(P, X, y):
    L = len(P) // 2
    logits, cache = mlp_forward(P, X)
    N = len(y)
    Pr = softmax_rows(logits)
    Y = np.eye(logits.shape[1])[y]
    D = (Pr - Y) / N                                   # ② Δ^(L) = (P − Y)/N
    G = {}
    for l in range(L, 0, -1):
        U_prev, Z_prev = cache[l - 1]
        G[f'W{l}'] = D.T @ Z_prev                      # ④ 勾配：δ × 入力側の活性
        G[f'b{l}'] = D.sum(axis=0)
        if l > 1:
            D = (D @ P[f'W{l}']) * (U_prev > 0)        # ③ 再帰：h'(u) ⊙ (W^T δ)
    return G

# 勾配チェック：小さなネットワークで全パラメータを数値微分と比べる
Xs = rng.normal(size=(5, 3))
ys = np.array([0, 2, 1, 2, 0])
P = init_mlp([3, 4, 3])
G = mlp_backward(P, Xs, ys)

def check(P, G, eps=1e-5):
    worst = 0.0
    for k in P:
        num = np.zeros_like(P[k])
        it = np.nditer(P[k], flags=['multi_index'])
        for _ in it:
            idx = it.multi_index
            old = P[k][idx]
            P[k][idx] = old + eps; Lp = mlp_loss(P, Xs, ys)
            P[k][idx] = old - eps; Lm = mlp_loss(P, Xs, ys)
            P[k][idx] = old
            num[idx] = (Lp - Lm) / (2 * eps)
        rel = np.abs(num - G[k]).max() / (np.abs(num).max() + np.abs(G[k]).max() + 1e-12)
        worst = max(worst, rel)
        print(f"{k}: 相対誤差 {rel:.2e}")
    return worst

worst = check(P, G)
print("最大の相対誤差 =", f"{worst:.2e}", "→ 1e-6 未満なら逆伝播は正しい" if worst < 1e-6 else "→ どこかが間違っている")

**やってみよう 3-1**　`mlp_backward` の再帰の行で `P[f'W{l}']` を転置（`.T`）にわざと変えて（形が合うように別の誤りでもよい），勾配チェックが失敗することを確かめる．
「勾配チェックが通る」ことの価値を体感する．

**やってみよう 3-2**　第2回の同心円データ（下のセルで再生成）を，この MLP と SGD で訓練して正解率を出す．

In [ ]:
n = 200
rr = np.concatenate([rng.uniform(0.0, 1.0, n), rng.uniform(1.6, 2.6, n)])
th = rng.uniform(0, 2 * np.pi, 2 * n)
Xc = np.stack([rr * np.cos(th), rr * np.sin(th)], axis=1)
yc = np.concatenate([np.zeros(n, int), np.ones(n, int)])

P = init_mlp([2, 16, 2])
eta = 0.5                            # TODO: 学習率を変える
hist = []
for step in range(2000):
    G = mlp_backward(P, Xc, yc)
    for k in P:
        P[k] -= eta * G[k]
    if step % 100 == 0:
        hist.append(mlp_loss(P, Xc, yc))
logits, _ = mlp_forward(P, Xc)
print("正解率 =", np.mean(logits.argmax(1) == yc))
plt.plot(np.arange(len(hist)) * 100, hist); plt.xlabel('step'); plt.ylabel('loss'); plt.grid(alpha=0.3); plt.show()

## §4 小さな自動微分（p.10–12）

「各演算は，受け取った勾配 × 自分の局所的な微分を入力側へ渡す」を，20 行ほどのクラスで実装する（micrograd 風）．
順伝播で計算グラフ（レシート）を記録し，`backward()` で逆順にたどる．

In [ ]:
class Value:
    def __init__(self, data, parents=(), local_grads=()):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents            # 入力側の Value
        self._local = local_grads          # 対応する局所的な微分 ∂self/∂parent

    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data + o.data, (self, o), (1.0, 1.0))

    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data * o.data, (self, o), (o.data, self.data))   # 乗算は「相手の値」を掛ける

    def __sub__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data - o.data, (self, o), (1.0, -1.0))

    def square(self):
        return Value(self.data ** 2, (self,), (2.0 * self.data,))

    def backward(self):
        # 出力側からたどる順序（トポロジカル順）を作る
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for p in v._parents:
                    visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0                     # ∂L/∂L = 1
        for v in reversed(order):
            for p, g in zip(v._parents, v._local):
                p.grad += v.grad * g        # 受け取った勾配 × 局所的な微分（同じ変数が複数回使われれば和）

w, x, b, y = Value(1.0), Value(2.0), Value(0.5), Value(4.0)
u = w * x
v = u + b
r = v - y
L = r.square()
L.backward()
print(f"u={u.data}, v={v.data}, r={r.data}, L={L.data}")
print(f"dL/dr={r.grad}, dL/dv={v.grad}, dL/du={u.grad}, dL/db={b.grad}, dL/dw={w.grad}, dL/dx={x.grad}")
# スライド p.11 と同じ：−3, −3, −3, −3, −6，そして p.11 のクイズ dL/dx = −3

**やってみよう 4-1（グラフの切断，p.12 の小話）**　途中で `float(v.data)` のように**値だけ**を取り出して続きを計算すると，
そこでグラフが切れて `w.grad` が 0 のままになる．エラーは出ない．

In [ ]:
w, x, b, y = Value(1.0), Value(2.0), Value(0.5), Value(4.0)
u = w * x
v = u + b
v_detached = Value(v.data)        # ← ここで値だけ取り出す（PyTorch の .item() / .detach() に相当）
r = v_detached - y
L = r.square()
L.backward()
print("dL/dw =", w.grad, "← 本当は −6 のはずが，切断されて 0．損失は計算できるが学習は動かない．")

**やってみよう 4-2（重み共有）**　同じ変数が 2 か所で使われると勾配は経路の和になる．$L=(w\cdot x + w)^2$ で $x=2$，$w=1$ のとき，
$\partial L/\partial w = 2(wx+w)(x+1) = 2\cdot3\cdot3 = 18$ になることを確かめる．

In [ ]:
w, x = Value(1.0), Value(2.0)
L = (w * x + w).square()
L.backward()
print("dL/dw =", w.grad)   # TODO: 18 になるか

## §5 SGD・Momentum・Adam の軌跡（p.13–15）

p.14 と同じ谷 $L=(w_1^2+12w_2^2)/2$ で，3 つの更新則の軌跡を描く．勾配は $\nabla L=(w_1, 12 w_2)$．

In [ ]:
H = np.array([1.0, 12.0])

def grad(w):
    return H * w

def sgd(w0, eta, steps):
    w = np.array(w0, float); path = [w.copy()]
    for _ in range(steps):
        w = w - eta * grad(w); path.append(w.copy())
    return np.array(path)

def momentum(w0, eta, alpha, steps):
    w = np.array(w0, float); v = np.zeros(2); path = [w.copy()]
    for _ in range(steps):
        v = alpha * v - eta * grad(w)
        w = w + v; path.append(w.copy())
    return np.array(path)

def adam(w0, eta, steps, b1=0.9, b2=0.999, eps=1e-8):
    w = np.array(w0, float); m = np.zeros(2); v = np.zeros(2); path = [w.copy()]
    for t in range(1, steps + 1):
        g = grad(w)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        mh = m / (1 - b1 ** t); vh = v / (1 - b2 ** t)     # バイアス補正
        w = w - eta * mh / (np.sqrt(vh) + eps); path.append(w.copy())
    return np.array(path)

w0 = (4.0, 1.5)
paths = {'SGD (eta=0.14)': sgd(w0, 0.14, 40),
         'Momentum (eta=0.14, alpha=0.7)': momentum(w0, 0.14, 0.7, 40),
         'Adam (eta=0.3)': adam(w0, 0.3, 40)}

g1, g2 = np.meshgrid(np.linspace(-2.5, 4.5, 200), np.linspace(-2, 2, 200))
Lgrid = 0.5 * (g1 ** 2 + 12 * g2 ** 2)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (name, p) in zip(axes, paths.items()):
    ax.contour(g1, g2, Lgrid, levels=[0.25, 0.5, 1, 2, 4, 8], colors='gray', linewidths=0.6)
    ax.plot(p[:, 0], p[:, 1], '-o', ms=3)
    ax.plot(p[-1, 0], p[-1, 1], 'ks', ms=6)
    ax.set_title(f"{name}\nfinal L={0.5 * (p[-1, 0] ** 2 + 12 * p[-1, 1] ** 2):.3f}")
    ax.set_aspect('equal')
plt.tight_layout(); plt.show()

**やってみよう 5-1**　Momentum の $\alpha$ を 0, 0.5, 0.9, 0.99 と変えると軌跡はどうなるか．$\alpha$ が大きすぎると何が起きるか．
**やってみよう 5-2**　SGD の $\eta$ を 0.17 にすると $w_2$ 方向（曲率 12）が発散する（$|1-12\eta|>1$）．Adam は同じ $\eta$ でも発散しない理由を p.15 のクイズから考える．

In [ ]:
for alpha in (0.0, 0.5, 0.9, 0.99):      # TODO
    p = momentum(w0, 0.14, alpha, 40)
    print(f"alpha={alpha}: 最終 L = {0.5 * (p[-1, 0] ** 2 + 12 * p[-1, 1] ** 2):.4f}")
p = sgd(w0, 0.17, 40); print("SGD eta=0.17: 最終 |w2| =", abs(p[-1, 1]))
p = adam(w0, 0.17, 40); print("Adam eta=0.17: 最終 L =", 0.5 * (p[-1, 0] ** 2 + 12 * p[-1, 1] ** 2))

## §6 学習率・スケジュール・クリッピング（p.16–17）

### 6.1 Warmup ＋ Cosine 減衰のスケジュールを描く

In [ ]:
def schedule(t, T_total=10000, T_warm=1000, eta_max=1.0):
    if t < T_warm:
        return eta_max * t / T_warm
    return eta_max * 0.5 * (1 + np.cos(np.pi * (t - T_warm) / (T_total - T_warm)))

ts = np.arange(10000)
plt.figure(figsize=(6, 3)); plt.plot(ts, [schedule(t) for t in ts]); plt.xlabel('step'); plt.ylabel('learning rate'); plt.grid(alpha=0.3); plt.show()

### 6.2 学習率の大小と Warmup の効果を §3 の MLP で見る

同心円データを，固定学習率（小・適・大）と Warmup＋Cosine で訓練し，損失曲線を比べる．

In [ ]:
def train(P, eta_fn, steps=1500, clip=None):
    hist = []
    for t in range(steps):
        G = mlp_backward(P, Xc, yc)
        if clip is not None:                                        # 勾配クリッピング（ノルム）
            norm = np.sqrt(sum((g ** 2).sum() for g in G.values()))
            fac = min(1.0, clip / (norm + 1e-12))
            G = {k: g * fac for k, g in G.items()}
        eta = eta_fn(t)
        for k in P:
            P[k] -= eta * G[k]
        hist.append(mlp_loss(P, Xc, yc))
    return np.array(hist)

settings = {'eta=0.02 (small)': lambda t: 0.02,
            'eta=0.5': lambda t: 0.5,
            'eta=3.0 (large)': lambda t: 3.0,
            'warmup+cosine (max 1.0)': lambda t: schedule(t, T_total=1500, T_warm=150, eta_max=1.0)}
plt.figure(figsize=(7, 3.8))
for name, fn in settings.items():
    rng_state = np.random.default_rng(1)           # 同じ初期値から始める
    P = {k: v.copy() for k, v in init_mlp([2, 16, 2]).items()}
    h = train(P, fn)
    plt.plot(h, label=f"{name}  final={h[-1]:.3f}")
plt.ylim(0, 1.0); plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()

### 6.3 勾配クリッピング（p.17）

$\boldsymbol{g}\leftarrow \boldsymbol{g}\cdot\min(1, c/\|\boldsymbol{g}\|)$．スライドの例 $\boldsymbol{g}=(3,4)$，$c=2.5$ で確かめ，
次に「たまに巨大な勾配が来る」状況を模擬して，クリッピングの有無で訓練がどう変わるかを見る．

In [ ]:
def clip_by_norm(g, c):
    return g * min(1.0, c / np.linalg.norm(g))

print("clip((3,4), 2.5) =", clip_by_norm(np.array([3.0, 4.0]), 2.5))    # (1.5, 2.0)

# 巨大な勾配のスパイクを模擬：200 ステップに 1 回，勾配を 50 倍にする
def train_spiky(P, eta=0.5, steps=1200, clip=None):
    hist = []
    for t in range(steps):
        G = mlp_backward(P, Xc, yc)
        if t % 200 == 199:
            G = {k: g * 50.0 for k, g in G.items()}
        if clip is not None:
            norm = np.sqrt(sum((g ** 2).sum() for g in G.values()))
            G = {k: g * min(1.0, clip / (norm + 1e-12)) for k, g in G.items()}
        for k in P:
            P[k] -= eta * G[k]
        hist.append(mlp_loss(P, Xc, yc))
    return np.array(hist)

plt.figure(figsize=(7, 3.6))
for clip in (None, 1.0):
    P = init_mlp([2, 16, 2])
    h = train_spiky(P, clip=clip)
    plt.plot(h, label=f"clip={clip}  final={h[-1]:.3f}")
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## §7 勾配消失の実験（p.17，第2回 p.7）

20 層の全結合ネットワーク（各層 64 ユニット）で，シグモイドと ReLU の場合に，**入力側の層ほど勾配が小さくなるか**を測る．
各層の $\|\boldsymbol{\delta}^{(l)}\|$（誤差信号の大きさ）を出力側から入力側へ逆伝播して記録する．

In [ ]:
def delta_norms(act, n_layers=20, width=64, scale=1.0):
    # 順伝播
    X = rng.normal(size=(32, width))
    Ws, Us, Z = [], [], X
    for l in range(n_layers):
        W = rng.normal(size=(width, width)) * scale / np.sqrt(width)
        U = Z @ W.T
        Z = sigmoid(U) if act == 'sigmoid' else relu(U)
        Ws.append(W); Us.append(U)
    # 逆伝播（出力層の誤差信号は乱数で与える．大きさの減衰だけを見る）
    D = rng.normal(size=Z.shape) / np.sqrt(Z.shape[1])
    norms = []
    for l in range(n_layers - 1, -1, -1):
        hp = sigmoid(Us[l]) * (1 - sigmoid(Us[l])) if act == 'sigmoid' else (Us[l] > 0)
        D = D * hp
        norms.append(np.linalg.norm(D) / np.sqrt(D.shape[0]))
        D = D @ Ws[l]
    return np.array(norms[::-1])    # 入力側 → 出力側の順

plt.figure(figsize=(6, 3.6))
for act, scale in (('sigmoid', 1.0), ('relu', np.sqrt(2.0))):     # ReLU は He 初期化（分散 2/n）
    ns = delta_norms(act, scale=scale)
    plt.semilogy(range(1, 21), ns, '-o', ms=3, label=act)
plt.xlabel('layer (1 = input side)'); plt.ylabel('||delta^(l)|| (log)'); plt.legend(); plt.grid(alpha=0.3); plt.show()

**やってみよう 7-1**　シグモイドで初期値のスケールを 4.0 にすると（大きな総入力 → 飽和），消失はさらに速くなるか．
**やってみよう 7-2**　ReLU で `scale=1.0`（Xavier 相当）と `sqrt(2)`（He 相当）を比べる．初期化がなぜ活性化関数に合わせて選ばれるのかを考える（第6回）．

In [ ]:
for act, scale in (('sigmoid', 4.0), ('relu', 1.0), ('relu', np.sqrt(2.0))):   # TODO
    ns = delta_norms(act, scale=scale)
    print(f"{act:8s} scale={scale:.2f}: 入力側の層/出力側の層 の比 = {ns[0] / ns[-1]:.2e}")

## §8（任意）PyTorch の autograd（付録B）

§4 の計算グラフを PyTorch で書く．`loss.backward()` が §4 の `backward()` に相当する．`.item()` で値を取り出すとグラフが切れることも見る．

In [ ]:
try:
    import torch
    w = torch.tensor(1.0, requires_grad=True)
    b = torch.tensor(0.5, requires_grad=True)
    x, y = torch.tensor(2.0), torch.tensor(4.0)
    L = (w * x + b - y) ** 2
    L.backward()
    print("dL/dw =", w.grad.item(), " dL/db =", b.grad.item())      # −6, −3

    # グラフの切断：.item() や .detach() を挟むと勾配は届かない
    w2 = torch.tensor(1.0, requires_grad=True)
    v = (w2 * x + b).item()             # ← ここで Python の float になりグラフが切れる
    L2 = torch.tensor((v - 4.0) ** 2)
    print("L2.requires_grad =", L2.requires_grad, "（False：backward しても w2.grad は None のまま）")
except ImportError:
    print("PyTorch が見つからないので，この節は読み飛ばす．")